# Step 1: Install Libraries

In [ ]:
#!pip install pandas
#!pip install polars
#!pip install pyarrow
#!pip install pyspark
#!pip install findspark
#!pip install dask
#!pip install requests
#!pip install httpx
#!pip install asyncio
#!pip install sqlalchemy
#!pip install psycopg2-binary
#!pip install apache-airflow
'''can not install because the system does not have windows long path support enabled '''
#!pip install prefect
#import sys
#!{sys.executable} -m pip install prefect
#!pip install great_expectations
#!pip install pandera 
#pip install apache-airflow


'can not install because the system does not have windows long path support enabled '

# Step 2: Import Libraries

In [2]:
import os
import pandas as pd
import polars as pl
from pyspark.sql import SparkSession
import findspark
import dask.dataframe as dd
import requests
import httpx
import asyncio
from sqlalchemy import create_engine
import psycopg2
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
#---------windows path length is breaking the installation-----------
#from prefect import flow, task 
import great_expectations as gx
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check


C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\airflow\__init__.py:45: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
2026-03-18T09:24:33.229936Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-03-18T09:24:33.237363Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-03-18T09:24:33.241290Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-03-18T09:24:33.246271Z [info     ] 

C:\Users\U1109200\AppData\Local\Temp\ipykernel_25284\325835088.py:13 DeprecatedImportWarning: The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`.

2026-03-18T09:24:33.560979Z [info     ] Skipping registering function get_context because it does not have a class [great_expectations._docs_decorators] loc=_docs_decorators.py:115
2026-03-18T09:24:34.864714Z [warning  ] C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 [py.warnings] filename=warnings.py lineno=110
2026-03-18T09:24:34.949962Z [info     ] Skipping registering function DataSourceManager._register_add_datasource.<locals>.crud_method_info because it is a closure [great_expectations._docs_decorators] loc=_docs_decorators.py:109
2026-03-18T09:24:34.957012Z [info     ] Skipping registering function DataSourceManager._register_update_datasource.<locals>.crud_method_info because it is 

# Step 3: Import Data

In [6]:
# Importing the sample data from csv file
df = pd.read_csv("../input_file/data.csv")

# Print dataframe (few rows)
df.head(10)
df.tail(10)
print('----------------------')

# Print the dataframe info
print('Info: \n',df.info())
print('----------------------')

# Print the dataframe details 
print('Describe:',df.describe())
print('----------------------')

# Print 
print('No. of columns',df.shape)

----------------------


<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        500 non-null    int64
 1   name      500 non-null    str  
 2   category  500 non-null    str  
 3   price     356 non-null    str  
 4   quantity  329 non-null    str  
 5   date      264 non-null    str  
dtypes: int64(1), str(5)
memory usage: 37.4 KB
Info: 
 None
----------------------
Describe:                id
count  500.000000
mean   250.500000
std    144.481833
min      1.000000
25%    125.750000
50%    250.500000
75%    375.250000
max    500.000000
----------------------
No. of columns (500, 6)


In [2]:
# Clean
df = df.drop_duplicates()

# Fix price column
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Remove invalid prices
df = df.dropna(subset=["price"])

NameError: name 'df' is not defined

# Step 4: Libraries Exploration

## Step 4.1: Pandas

In [ ]:
df = pd.read_csv("../input_file/data.csv")

# Clean
df = df.dropna()
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Transform
summary = df.groupby("category")["price"].mean()

print(summary)

category
Clothing        825.000000
Electronics    1200.000000
Grocery         216.666667
Name: price, dtype: float64


In [7]:
df1 = df
df1 = df1.drop_duplicates()
df1

,id,name,category,price,quantity,date
0,1,Product_A,Clothing,invalid,9,03-01-2024
1,2,Product_B,Electronics,invalid,NaN,01-02-2024
2,3,Product_C,Clothing,invalid,NaN,NaN
3,4,Product_D,Clothing,invalid,ten,21-01-2024
4,5,Product_E,Grocery,invalid,9,28-01-2024
...,...,...,...,...,...,...
495,496,Product_B,Grocery,1441,6,NaN
496,497,Product_C,Clothing,NaN,6,01-01-2024
497,498,Product_D,Clothing,185,6,NaN
498,499,Product_E,Clothing,invalid,3,NaN


## Step 4.2: polars

In [8]:
df2 = df
df2 = df2.drop_duplicates()
df2

,id,name,category,price,quantity,date
0,1,Product_A,Clothing,invalid,9,03-01-2024
1,2,Product_B,Electronics,invalid,NaN,01-02-2024
2,3,Product_C,Clothing,invalid,NaN,NaN
3,4,Product_D,Clothing,invalid,ten,21-01-2024
4,5,Product_E,Grocery,invalid,9,28-01-2024
...,...,...,...,...,...,...
495,496,Product_B,Grocery,1441,6,NaN
496,497,Product_C,Clothing,NaN,6,01-01-2024
497,498,Product_D,Clothing,185,6,NaN
498,499,Product_E,Clothing,invalid,3,NaN


In [ ]:
df = pl.read_csv("../input_file/data.csv")

result = (
    df.drop_nulls()
      .group_by("category")
      .agg(pl.col("price").mean())
)

print(result)

shape: (3, 2)
┌─────────────┬───────┐
│ category    ┆ price │
│ ---         ┆ ---   │
│ str         ┆ str   │
╞═════════════╪═══════╡
│ Electronics ┆ null  │
│ Grocery     ┆ null  │
│ Clothing    ┆ null  │
└─────────────┴───────┘


# Big Data Processing
## - PySpark

In [6]:

df = pd.read_csv("../input_file/data.csv")

# Clean price column
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Group and average
result = df.groupby("category")["price"].mean()

print(result)

category
Clothing       1286.898734
Electronics    1421.465517
Grocery        1390.403846
Name: price, dtype: float64


# Big Data Processing
## - Dask

In [ ]:


df = dd.read_csv("../input_file/data.csv")

# Clean data
df["price"] = dd.to_numeric(df["price"], errors="coerce")

# Group and aggregate
result = df.groupby("category").agg({"price": "mean"}).compute()

print(result)

                   price
category                
Clothing     1286.898734
Electronics  1421.465517
Grocery      1390.403846


# Data Ingestion
## - Requests

In [17]:
import requests

res = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    verify=False
)

print(res.json()[:2])

[{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}, {'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}]


C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'jsonplaceholder.typicode.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


# Data Ingestion 
## - HTTPX (Async)

In [23]:
import httpx

async def fetch():
    async with httpx.AsyncClient(
        timeout=10,
        verify=False   # ⚠️ bypass SSL issues in your network
    ) as client:
        res = await client.get("https://jsonplaceholder.typicode.com/posts")
        return res.json()

data = await fetch()
print(data[:2])

[{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}, {'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}]


# Databases
## - SQLAlchemy

In [ ]:


engine = create_engine("sqlite:///test.db")

df.to_sql("table_name", engine, if_exists="replace", index=False)

print("Done")

Done


# Databases
## - psycopg2

In [36]:
import psycopg2

# 1. Connect to database
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="your_password",
    host="localhost",
    port="5432"
)

cur = conn.cursor()

# 2. Create table
cur.execute("""
CREATE TABLE IF NOT EXISTS test (
    id INT,
    name TEXT
)
""")

# 3. Insert data
cur.execute("""
INSERT INTO test (id, name)
VALUES (%s, %s)
""", (1, "Alice"))

# 4. Read data
cur.execute("SELECT * FROM test")
rows = cur.fetchall()

print(rows)

# 5. Save changes
conn.commit()

# 6. Close
cur.close()
conn.close()

OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


# Orchestration
## - Airflow (DAG)

In [1]:
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime

def task1():
    print("Task 1: Extract data")

def task2():
    print("Task 2: Process data")

def task3():
    print("Task 3: Load data")

with DAG(
    dag_id="simple_pipeline",
    start_date=datetime(2024, 1, 1),
    schedule="@daily",   # ✅ FIXED
    catchup=False
) as dag:

    t1 = PythonOperator(task_id="extract", python_callable=task1)
    t2 = PythonOperator(task_id="process", python_callable=task2)
    t3 = PythonOperator(task_id="load", python_callable=task3)

    t1 >> t2 >> t3

C:\Users\U1109200\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\airflow\__init__.py:45: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
2026-03-21T14:36:19.522841Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-03-21T14:36:19.530442Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-03-21T14:36:19.535323Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-03-21T14:36:19.541805Z [info     ] 

# Orchestration
## - Perfect

In [ ]:
#-------can not run because the libraries are unable to download in the system---------
@task
def clean():
    return "cleaned"

@flow
def pipeline():
    clean()

pipeline()

ModuleNotFoundError: No module named 'prefect'

# Data Validation
## - Great Expectations

In [74]:
import pandas as pd
import great_expectations as gx

# -----------------------------
# Step 1: Load data
# -----------------------------
df = pd.read_csv("../input_file/data.csv")

# -----------------------------
# Step 2: Get context
# -----------------------------
context = gx.get_context()

# -----------------------------
# Step 3: Create datasource
# -----------------------------
datasource = context.sources.add_pandas("my_source")

# -----------------------------
# Step 4: Create data asset
# -----------------------------
data_asset = datasource.add_dataframe_asset(name="my_asset")

# -----------------------------
# Step 5: Build batch
# -----------------------------
batch_request = data_asset.build_batch_request(dataframe=df)

# -----------------------------
# Step 6: Get validator
# -----------------------------
validator = context.get_validator(batch_request=batch_request)

# -----------------------------
# Step 7: Expectations
# -----------------------------
validator.expect_column_values_to_not_be_null("price")
validator.expect_column_values_to_be_between("price", min_value=0)

# -----------------------------
# Step 8: Validate
# -----------------------------
results = validator.validate()

print(results)

2026-03-18T06:26:25.555497Z [info     ] Could not find local file-backed GX project [great_expectations.data_context.data_context.context_factory] loc=context_factory.py:398
2026-03-18T06:26:25.568621Z [info     ] Created temporary directory 'C:\Users\U1109200\AppData\Local\Temp\tmp2_g8mska' for ephemeral docs site [great_expectations.data_context.types.base] loc=base.py:1401
2026-03-18T06:26:25.579766Z [info     ] Loading 'datasources' ->
[]    [great_expectations.datasource.fluent.config] loc=config.py:191


AttributeError: 'EphemeralDataContext' object has no attribute 'sources'

# Data Validation
## - Pandera

In [63]:

# -----------------------------
# Step 1: Load Data
# -----------------------------
df = pd.read_csv("../input_file/data.csv")

# -----------------------------
# Step 2: Inspect (DON’T SKIP)
# -----------------------------
print("Before cleaning:")
print(df["price"].head(10))
print(df["price"].dtype)

# -----------------------------
# Step 3: Clean Data
# -----------------------------
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(",", "", regex=True)
    .str.replace("₹", "", regex=True)
    .str.strip()
)

# Convert to numeric
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# -----------------------------
# Step 4: Handle Missing Values
# -----------------------------
# Choose ONE strategy

# Option A (recommended)
df = df.dropna(subset=["price"])

# Option B (if you must keep rows)
# df["price"] = df["price"].fillna(df["price"].median())

# -----------------------------
# Step 5: Define Schema
# -----------------------------
schema = DataFrameSchema({
    "price": Column(
        float,
        Check.ge(0),   # price >= 0
        nullable=False
    )
})

# -----------------------------
# Step 6: Validate
# -----------------------------
validated_df = schema.validate(df)

print("Validation successful ✅")
print(validated_df.head())

Before cleaning:
0    invalid
1    invalid
2    invalid
3    invalid
4    invalid
5    invalid
6    invalid
7       1499
8    invalid
9       1144
Name: price, dtype: str
str
Validation successful ✅
    id       name  category   price quantity        date
7    8  Product_H  Clothing  1499.0      ten  22-02-2024
9   10  Product_J  Clothing  1144.0      NaN  22-02-2024
13  14  Product_N   Grocery   868.0        6         NaN
17  18  Product_R   Grocery  2343.0      NaN  02-02-2024
19  20  Product_T  Clothing   627.0      ten  03-01-2024


# Data Transformation
## - pyjanitor

In [ ]:
import janitor

df = df.clean_names().remove_empty()

# Data Transformation
## - toolz

In [ ]:
from toolz import pipe

result = pipe(
    [1,2,3,4],
    lambda x: filter(lambda i: i > 2, x),
    list
)

print(result)

# Machine Learning
## - scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

# Machine Learning
## - XGBoost

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor()
model.fit(X, y)

# Visualization
## - matplotlib

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df["price"])
plt.show()

# Visualization
## - plotly

In [ ]:
import plotly.express as px

fig = px.line(df, x="date", y="price")
fig.show()

# Time Series
## - statsmodels

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(df["price"], order=(1,1,1))
model_fit = model.fit()

# Time Series
## - prophet 

In [ ]:
from prophet import Prophet

df.columns = ["ds", "y"]

model = Prophet()
model.fit(df)

# Performance
## - multiprocessing

In [ ]:
from multiprocessing import Pool

def square(x):
    return x*x

with Pool(4) as p:
    print(p.map(square, [1,2,3]))

# Perfmance 
## - numba 

In [ ]:
from numba import jit

@jit
def compute(x):
    return x**2

print(compute(10))

# Logging
## - loguru

In [ ]:
from loguru import logger

logger.info("Pipeline started")

# Logging 
## - Prometheus

In [ ]:
from prometheus_client import Counter

c = Counter("requests", "Total requests")
c.inc()

# Configuration
## - dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("DB_URL"))

# Configuration
## - hydra 

In [ ]:
import hydra

@hydra.main(config_path=".", config_name="config")
def main(cfg):
    print(cfg.db)

main()

# Testing
## - pytest

In [ ]:
def test_sum():
    assert sum([1,2]) == 3

# Testing 
## - hypothesis

In [ ]:
from hypothesis import given
import hypothesis.strategies as st

@given(st.integers())
def test_num(x):
    assert x == x

# NLP
## - spaCy

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is a company")

for ent in doc.ents:
    print(ent.text, ent.label_)

# NLP
## - Transformers 

In [ ]:
from transformers import pipeline

ner = pipeline("ner")
print(ner("OpenAI builds AI"))